# User-specific training data levels

XGBoost trained on Level 1, Level 2 or Level 3 user-specific data and tested on a fixed test fold under
cross-user mixing with stratified 10-fold cross-validation.

- Level 1: data of the test users from other VR scenarios
- Level 2: data of the test users from the same scenario but other 11-s segments
- Level 3: data of the test users from the same 11-s segments (intra-segment calibration)

Input: `../data/window_features.csv`. Output: per-fold and averaged results in `results/`.

In [ ]:
import os
import warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

DATA_PATH = os.path.join("..", "data", "window_features.csv")
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

TEST_SCENARIOS_PER_USER = 3

USER_COL = "participant"
SCENARIO_COL = "simulation"
SEGMENT_COL = "segment_marker"
LABEL_COL = "cs"

feature_cols_raw = [
    "LeftPupilPosInSensorX", "LeftPupilPosInSensorY", "RightPupilPosInSensorX", "RightPupilPosInSensorY",
    "GazeOriginLclSpc_X", "GazeOriginLclSpc_Y", "GazeOriginLclSpc_Z",
    "GazeOriginWrldSpc_X", "GazeOriginWrldSpc_Y", "GazeOriginWrldSpc_Z",
    "NrmLeftEyeOriginX", "NrmLeftEyeOriginY", "NrmLeftEyeOriginZ",
    "NrmRightEyeOriginX", "NrmRightEyeOriginY", "NrmRightEyeOriginZ",
    "HeadQRotationX", "HeadQRotationY", "HeadQRotationZ", "HeadQRotationW",
    "HeadEulX", "HeadEulY", "HeadEulZ",
]

In [ ]:
def build_xgb_model(num_class):
    return make_pipeline(
        StandardScaler(),
        XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                      gamma=0.1, reg_alpha=0.1, objective="multi:softprob", eval_metric="mlogloss",
                      num_class=int(num_class), random_state=SEED, n_jobs=-1),
    )


def evaluate_and_get_metrics(X_train, y_train, X_test, y_test):
    """Oversample the training set, train XGBoost and return Acc and per-class P/R/F1 (rounded to 2 decimals)."""
    ros = RandomOverSampler(random_state=SEED)
    try:
        X_train, y_train = ros.fit_resample(X_train, y_train)
    except ValueError:
        return None
    if len(np.unique(y_train)) < 2:
        return None

    # XGBoost needs consecutive labels 0..K-1: map the training labels and map predictions back.
    unique_train = np.sort(np.unique(y_train))
    label_to_compact = {lab: i for i, lab in enumerate(unique_train)}
    compact_to_label = {i: lab for i, lab in enumerate(unique_train)}
    y_train_compact = np.vectorize(label_to_compact.get)(y_train).astype(int)

    model = build_xgb_model(num_class=len(unique_train))
    model.fit(X_train, y_train_compact)

    y_test_1d = np.asarray(y_test).reshape(-1).astype(int)
    y_pred_compact = np.asarray(model.predict(X_test))
    if y_pred_compact.ndim == 2:
        y_pred_compact = (y_pred_compact.reshape(-1) if y_pred_compact.shape[1] == 1
                          else np.argmax(y_pred_compact, axis=1))
    y_pred = np.vectorize(compact_to_label.get)(y_pred_compact.reshape(-1).astype(int))
    y_pred = np.asarray(y_pred).reshape(-1).astype(int)

    acc = accuracy_score(y_test_1d, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_test_1d, y_pred, labels=[0, 1, 2, 3], average=None,
                                                 zero_division=0)
    res = OrderedDict()
    res["ACC"] = round(float(acc), 2)
    for i in range(4):
        res[f"PRECISION{i}"] = round(float(p[i]), 2)
    for i in range(4):
        res[f"RECALL{i}"] = round(float(r[i]), 2)
    for i in range(4):
        res[f"F1_{i}"] = round(float(f[i]), 2)
    return res

In [ ]:
df = pd.read_csv(DATA_PATH).dropna(how="all")
feature_cols = [f"{c}__mean" if f"{c}__mean" in df.columns else c for c in feature_cols_raw
                if (c in df.columns or f"{c}__mean" in df.columns)]
X = df[feature_cols].values
y = df[LABEL_COL].astype(int).values

LEVELS = ["L1", "L2", "L3"]
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
all_fold_results = []

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    df_test = df.iloc[test_idx]
    df_candidates = df.iloc[train_idx]

    rng = np.random.RandomState(SEED + fold_idx)
    keep_idx = []
    for u, g in df_test.groupby(USER_COL):
        scen_list = pd.unique(g[SCENARIO_COL])
        chosen = (rng.choice(scen_list, size=TEST_SCENARIOS_PER_USER, replace=False)
                  if len(scen_list) > TEST_SCENARIOS_PER_USER else scen_list)
        keep_idx.extend(g[g[SCENARIO_COL].isin(chosen)].index.tolist())
    keep_idx = sorted(set(keep_idx))
    dropped_idx = sorted(set(df_test.index) - set(keep_idx))
    df_test = df.loc[keep_idx]
    if len(dropped_idx) > 0:
        df_candidates = pd.concat([df_candidates, df.loc[dropped_idx]], axis=0)

    u_test = df_test[USER_COL].unique()
    user_scenario_map = df_test.groupby(USER_COL)[SCENARIO_COL].unique().to_dict()
    user_segment_map = df_test.groupby(USER_COL)[SEGMENT_COL].unique().to_dict()

    l1_mask = df_candidates.apply(
        lambda row: row[USER_COL] in u_test
        and row[SCENARIO_COL] not in user_scenario_map.get(row[USER_COL], []), axis=1)
    l2_mask = df_candidates.apply(
        lambda row: row[USER_COL] in u_test
        and row[SCENARIO_COL] in user_scenario_map.get(row[USER_COL], [])
        and row[SEGMENT_COL] not in user_segment_map.get(row[USER_COL], []), axis=1)
    l3_mask = df_candidates.apply(
        lambda row: row[USER_COL] in u_test
        and row[SCENARIO_COL] in user_scenario_map.get(row[USER_COL], [])
        and row[SEGMENT_COL] in user_segment_map.get(row[USER_COL], []), axis=1)
    levels = OrderedDict([("L1", df_candidates[l1_mask]), ("L2", df_candidates[l2_mask]),
                          ("L3", df_candidates[l3_mask])])

    fold_log = OrderedDict([("Fold", fold_idx + 1), ("FixedTestSet_Size", len(df_test))])
    for lvl in LEVELS:
        fold_log[f"{lvl}_DataSize"] = len(levels[lvl])
    for lvl in LEVELS:
        fold_log[f"{lvl}_ACC"] = None

    X_test_f, y_test_f = df_test[feature_cols].values, df_test[LABEL_COL].astype(int).values
    for lvl_name, train_df in levels.items():
        unique_classes = train_df[LABEL_COL].nunique() if not train_df.empty else 0
        if len(train_df) >= 10 and unique_classes >= 2:
            X_tr, y_tr = train_df[feature_cols].values, train_df[LABEL_COL].astype(int).values
            metrics = evaluate_and_get_metrics(X_tr, y_tr, X_test_f, y_test_f)
            if metrics:
                for k, v in metrics.items():
                    fold_log[f"{lvl_name}_{k}"] = v

    all_fold_results.append(fold_log)

## Summary (mean over the 10 folds)

In [ ]:
df_final = pd.DataFrame(all_fold_results)
OUT_COLS = (["ACC"] + [f"PRECISION{i}" for i in range(4)] + [f"RECALL{i}" for i in range(4)]
            + [f"F1_{i}" for i in range(4)])
rows = []
for lvl in LEVELS:
    row = OrderedDict(level=lvl, sample_size=df_final[f"{lvl}_DataSize"].mean())
    for metric in OUT_COLS:
        col_name = f"{lvl}_{metric}"
        row[metric] = (float(pd.to_numeric(df_final[col_name], errors="coerce").mean())
                       if col_name in df_final.columns else np.nan)
    rows.append(row)
summary = pd.DataFrame(rows).round(2)

df_final.to_csv(os.path.join(OUT_DIR, "levels_folds.csv"), index=False)
summary.to_csv(os.path.join(OUT_DIR, "levels_summary.csv"), index=False)
print(summary.to_string(index=False))